# Visualisation du dataset tomate

Notebook d'exploration du dataset construit par `build_tomato_dataset.py`.

## Comment utiliser ce notebook

### Prérequis

Le script `build_tomato_dataset.py` doit avoir été exécuté avec succès. Il génère dans `<data_dir>/tomato_only/` : les CSV de split, les class weights, l'arborescence `tomato_final/` et l'archive zip. Sans ces artefacts, aucune cellule de ce notebook ne tournera.

Commande de référence :

```bash
python build_tomato_dataset.py --data-dir ./dataset
```

### Environnement — cohérence avec le script

**Important** : le kernel Jupyter doit tourner dans le même environnement que celui qui a exécuté le script. Sinon les chemins stockés dans les CSV ne seront pas résolvables.

- Si le script a été lancé depuis **WSL** (chemins de la forme `/mnt/c/...` dans les CSV), le notebook doit aussi tourner en kernel WSL. Lancez-le depuis le shell WSL :
```bash
  jupyter notebook --no-browser
  jupyter notebook visualize_tomato_dataset.ipynb
```
  Puis copiez l'URL `http://localhost:8888/...` dans votre navigateur Windows.

- Si le script a été lancé depuis **Windows natif** (chemins de la forme `C:\Users\...`), utilisez un kernel Windows.

Pour vérifier le kernel actuel :
```python
import sys, platform
print(sys.executable)
print(platform.system())
```

### Plan du notebook

1. Configuration et inspection de l'arborescence
2. Distribution des classes (global, avant split)
3. Échantillons d'images par classe + analyse qualitative
4. Vérification des tailles d'image
5. Distribution train / val / test
6. Visualisation des augmentations
7. Class weights
8. Vérification finale d'intégrité (chemins, chevauchements, cohérence)

### Dépendances Python

```bash
pip install pandas numpy matplotlib pillow jupyter
```

## 1. Configuration

In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# >>> À ADAPTER À VOTRE ENVIRONNEMENT <<<
DATA_DIR = Path("/mnt/c/Users/LARIS-1/Desktop/DJMA/tomato_classification/data/dataset")
OUTPUT_DIR = DATA_DIR / "tomato_only"
AUG_DIR    = OUTPUT_DIR / "augmented_train"
FINAL_DIR  = OUTPUT_DIR / "tomato_final"

assert OUTPUT_DIR.exists(), f"OUTPUT_DIR introuvable : {OUTPUT_DIR}"
print("OUTPUT_DIR :", OUTPUT_DIR)
print("AUG_DIR    :", AUG_DIR, "[ok]" if AUG_DIR.exists() else "[absent]")
print("FINAL_DIR  :", FINAL_DIR, "[ok]" if FINAL_DIR.exists() else "[absent]")

### Arborescence du dossier fusionné

In [ ]:
def print_tree_dirs_only(start_path: Path, prefix: str = "", max_depth: int = 3, depth: int = 0) -> None:
    if depth >= max_depth:
        return
    dirs = sorted([d for d in start_path.iterdir() if d.is_dir()])
    for i, d in enumerate(dirs):
        last = i == len(dirs) - 1
        print(prefix + ("└── " if last else "├── ") + d.name)
        print_tree_dirs_only(d, prefix + ("    " if last else "│   "), max_depth, depth + 1)

print(f"📁 {OUTPUT_DIR.name}")
print_tree_dirs_only(OUTPUT_DIR, max_depth=2)

## 2. Distribution des classes (global, avant split)

On exclut les dossiers techniques `augmented_train/` et `tomato_final/` qui ne sont pas des classes.

In [ ]:
TECHNICAL_DIRS = {"augmented_train", "tomato_final"}

def count_images_by_class(root: Path) -> dict[str, int]:
    counts = {}
    for class_dir in sorted(root.iterdir()):
        if not class_dir.is_dir() or class_dir.name in TECHNICAL_DIRS:
            continue
        n = sum(1 for f in class_dir.iterdir()
                if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
        counts[class_dir.name] = n
    return counts

class_counts = count_images_by_class(OUTPUT_DIR)
total = sum(class_counts.values())

print(f"Nombre total de classes : {len(class_counts)}")
print(f"Nombre total d'images   : {total}\n")
for c, n in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {c:<45} {n:>5}")

In [ ]:
items = sorted(class_counts.items(), key=lambda x: x[1])
classes = [c for c, _ in items]
counts  = [n for _, n in items]

fig, ax = plt.subplots(figsize=(12, max(4, 0.35 * len(classes))))
bars = ax.barh(classes, counts, color="#4A90A4")
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + max(counts) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            str(count), va="center", fontsize=9)
ax.set_xlabel("Nombre d'images")
ax.set_title("Distribution des classes — dataset tomate fusionné")
plt.tight_layout()
plt.show()

### Analyse

Une base dédiée à la classification des maladies de la tomate a été construite à partir de plusieurs jeux de données publics. Les images relatives à la tomate ont été extraites, les noms de classes normalisés, les doublons exacts et les images corrompues supprimés, puis certaines classes sémantiquement redondantes ont été fusionnées (ex. `tomato_leaf_curl` → `tomato_yellow_leaf_curl_virus`). La base finale contient 12 classes et ~24 000 images.

## 3. Échantillons par classe

4 images aléatoires par classe — inspection visuelle pour repérer les biais de domaine, les classes hétérogènes et les symptômes discriminants.

In [ ]:
N_PER_CLASS = 4
random.seed(42)

images_by_class = {}
for class_name in sorted(class_counts.keys()):
    class_path = OUTPUT_DIR / class_name
    files = [f for f in class_path.iterdir()
             if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
    if files:
        images_by_class[class_name] = files

n_classes = len(images_by_class)
fig, axes = plt.subplots(
    n_classes, N_PER_CLASS,
    figsize=(3 * N_PER_CLASS, 2.5 * n_classes),
    squeeze=False,
)

for row, (class_name, files) in enumerate(images_by_class.items()):
    selected = random.sample(files, min(N_PER_CLASS, len(files)))
    for col in range(N_PER_CLASS):
        ax = axes[row][col]
        if col < len(selected):
            try:
                with Image.open(selected[col]) as im:
                    ax.imshow(im.convert("RGB"))
            except Exception as e:
                ax.text(0.5, 0.5, f"Erreur\n{e}", ha="center", va="center")
        if col == 0:
            ax.set_ylabel(class_name, fontsize=9, rotation=0,
                          ha="right", va="center", labelpad=10)
        ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("Échantillons aléatoires par classe", fontsize=14, y=1.001)
plt.tight_layout()
plt.show()

### Analyse qualitative des images

**Biais de domaine (images contrôlées vs images réelles)**

On observe que beaucoup d'images présentent des feuilles isolées, un fond uniforme gris, blanc ou noir, une prise de vue très proche et peu de contexte autour de la plante. Conséquence : le modèle risque d'être bon sur ce type d'images « contrôlées », mais moins bon sur des photos réelles d'application mobile (plante entière, fond complexe, lumière variable, plusieurs feuilles visibles, flou, ombres, terre, main, pot, etc.). **C'est une première faiblesse majeure : manque de robustesse au changement de domaine.**

---

**Confusion probable entre maladies foliaires proches**

Certaines classes sont définies par des signes très fins, pas par une forme globale. Donc, même avec une bonne accuracy globale, le modèle peut échouer sur les cas où les indices visuels fins sont peu visibles.

Il existe un fort risque de confusion entre les classes à nécroses foliaires. En regardant la grille d'images, plusieurs classes partagent : brunissement, nécrose, dessèchement, bord de feuille abîmé, lésions irrégulières. C'est particulièrement visible pour : `tomato_early_blight`, `tomato_late_blight`, `tomato_leaf_blight`, `tomato_bacterial_spot`, `tomato_verticillium_wilt`.

Lacune probable : le modèle apprendra peut-être surtout un signal du type **« feuille très malade / nécrosée »** au lieu d'apprendre quelle maladie foliaire précise. Autrement dit, il peut bien séparer sain vs malade, mais moins bien séparer maladie A vs maladie B quand les symptômes sont proches.

---

**Variabilité intra-classe liée aux stades d'atteinte**

Plusieurs classes semblent contenir des stades différents d'atteinte. Dans certaines classes on voit des feuilles faiblement touchées, d'autres très dégradées, parfois presque entièrement nécrosées. Conséquence : la variabilité intra-classe est forte, le modèle devra apprendre qu'une même maladie peut apparaître sous plusieurs formes.

---

**Classe `tomato_healthy` non homogène**

Les feuilles ne se ressemblent pas toutes parfaitement : certaines sont nettes et lumineuses, d'autres ont un aspect plus sombre ou texturé. Conséquence : le modèle peut apprendre une frontière floue entre sain et légèrement altéré — risque de faux positifs sur des feuilles saines mais atypiques, ou de faux négatifs si une feuille malade est peu symptomatique.

---

**Cas de `tomato_yellow_leaf_curl_virus`**

Cette classe présente une forte variabilité intra-classe liée au stade d'infection. Les images révèlent un spectre allant d'une feuille quasi saine visuellement (stade précoce, vert uniforme, nervures peu contrastées) à une feuille plus atteinte (jaunissement inter-nervures dominant, limbe cloqué, bords crispés). Le vrai marqueur discriminant n'est pas la couleur globale mais le pattern nervures claires / limbe foncé associé à une déformation du limbe. Ce signal est fin, localisé.

Conséquence : les images de stade précoce — où ce signal est absent ou très faible — constituent un bruit d'apprentissage actif. Le modèle ne peut pas apprendre un marqueur qu'il ne voit pas, et associera ces images à la classe `healthy` ou à une autre classe.

---

**Synthèse — risques attendus**

1. Confusions entre maladies foliaires proches
2. Mauvaise généralisation hors dataset (photos de terrain)
3. Sensibilité aux cas atypiques (stades précoces, feuilles saines texturées)

## 4. Vérification des dimensions d'images

Utile pour anticiper le `Resize` à appliquer en amont du modèle.

In [ ]:
sizes = []
for class_name in class_counts:
    for f in (OUTPUT_DIR / class_name).iterdir():
        if f.is_file() and f.suffix.lower() in IMAGE_EXTS:
            try:
                with Image.open(f) as im:
                    sizes.append(im.size)
            except Exception:
                pass

size_series = pd.Series(sizes, name="size")
print(f"Nombre de tailles distinctes : {size_series.nunique()}")
print("\nTop 10 des tailles les plus fréquentes :")
print(size_series.value_counts().head(10))

## 5. Vérification des images à fond noir retirées

Le script a déplacé les images détectées comme "fond noir" vers `<data_dir>/black_background_removed/`, en préservant l'arborescence par classe. Cette section vérifie visuellement :

1. **Combien** d'images ont été déplacées, par classe
2. **Ce qui a été déplacé** — s'assurer qu'il n'y a pas de faux positifs (feuilles nécrosées classées à tort)
3. **Ce qui est resté dans `tomato_only/`** — s'assurer qu'on n'a pas raté de fonds noirs évidents

Si vous voyez des faux positifs ou des faux négatifs, ajustez les seuils `black_bg_*` dans `PipelineConfig` et relancez le pipeline.

In [ ]:
BLACK_BG_DIR = DATA_DIR / "black_background_removed"

if not BLACK_BG_DIR.exists():
    print(f"Pas de dossier {BLACK_BG_DIR.name} — aucune image à fond noir n'a été détectée.")
    black_bg_files = []
else:
    black_bg_files = []
    for class_dir in sorted(BLACK_BG_DIR.iterdir()):
        if not class_dir.is_dir():
            continue
        for f in class_dir.iterdir():
            if f.is_file() and f.suffix.lower() in IMAGE_EXTS:
                black_bg_files.append((class_dir.name, f))

    # Compte par classe
    from collections import Counter
    counts = Counter(cls for cls, _ in black_bg_files)
    print(f"Total images à fond noir déplacées : {len(black_bg_files)}\n")
    print("Par classe :")
    for cls, n in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"  {cls:<45} {n:>4}")

In [ ]:
# visualisation des images déplacées (faux positifs ?)
if black_bg_files:
    random.seed(42)
    sample_moved = random.sample(black_bg_files, min(12, len(black_bg_files)))

    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    for ax, (cls, path) in zip(axes.flat, sample_moved):
        with Image.open(path) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(cls, fontsize=9)
        ax.axis("off")
    for ax in axes.flat[len(sample_moved):]:
        ax.axis("off")
    plt.suptitle("Échantillon d'images DÉPLACÉES — vérifier l'absence de faux positifs",
                 fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("Rien à visualiser.")

### Contrôle inverse : échantillon d'images *non* déplacées

Si des fonds noirs évidents apparaissent ici, les seuils sont trop stricts — baissez `black_bg_dark_ratio` (ex: de 0.85 à 0.75) ou montez `black_bg_dark_threshold` (ex: de 30 à 40) dans `PipelineConfig`, puis relancez le pipeline.

In [ ]:
# visualisation des images restantes (faux négatifs ?)
# Échantillon aléatoire de ce qui est RESTÉ dans tomato_only/
remaining = []
for class_dir in sorted(OUTPUT_DIR.iterdir()):
    if not class_dir.is_dir() or class_dir.name in TECHNICAL_DIRS:
        continue
    for f in class_dir.iterdir():
        if f.is_file() and f.suffix.lower() in IMAGE_EXTS:
            remaining.append((class_dir.name, f))

sample_kept = random.sample(remaining, min(12, len(remaining)))

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, (cls, path) in zip(axes.flat, sample_kept):
    with Image.open(path) as im:
        ax.imshow(im.convert("RGB"))
    ax.set_title(cls, fontsize=9)
    ax.axis("off")
plt.suptitle("Échantillon d'images CONSERVÉES — vérifier qu'aucun fond noir n'a échappé",
             fontsize=13)
plt.tight_layout()
plt.show()


## 6. Distribution train / val / test

Lecture des trois CSV de split produits par le pipeline.

In [ ]:
train_df = pd.read_csv(OUTPUT_DIR / "train_split_final.csv")
val_df   = pd.read_csv(OUTPUT_DIR / "val_split_final.csv")
test_df  = pd.read_csv(OUTPUT_DIR / "test_split_final.csv")

print(f"train : {len(train_df)}  (incluant augmentations)")
print(f"val   : {len(val_df)}")
print(f"test  : {len(test_df)}")

In [ ]:
train_counts = train_df["class_name"].value_counts().rename("train")
val_counts   = val_df["class_name"].value_counts().rename("val")
test_counts  = test_df["class_name"].value_counts().rename("test")

split_table = (
    pd.concat([train_counts, val_counts, test_counts], axis=1)
    .fillna(0).astype(int)
    .sort_index()
)
split_table["total"] = split_table.sum(axis=1)
for col in ["train", "val", "test"]:
    split_table[f"{col}_%"] = (split_table[col] / split_table["total"] * 100).round(2)

totals = split_table[["train", "val", "test", "total"]].sum()
split_table.loc["TOTAL"] = {
    "train":   int(totals["train"]),
    "val":     int(totals["val"]),
    "test":    int(totals["test"]),
    "total":   int(totals["total"]),
    "train_%": round(totals["train"] / totals["total"] * 100, 2),
    "val_%":   round(totals["val"]   / totals["total"] * 100, 2),
    "test_%":  round(totals["test"]  / totals["total"] * 100, 2),
}
split_table

In [ ]:
plot_df = split_table.drop(index="TOTAL", errors="ignore")[["train", "val", "test"]]

ax = plot_df.plot(
    kind="bar",
    figsize=(14, 6),
    color=["#4A90A4", "#F5A623", "#D0021B"],
)
ax.set_title("Distribution finale par classe : train / val / test")
ax.set_xlabel("Classe")
ax.set_ylabel("Nombre d'images")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Analyse

Le découpage initial a été réalisé de manière stratifiée selon les régimes 70/15/15 et 50/25/25 (pour les classes < 1000 images). Ensuite, seul l'ensemble d'entraînement a été modifié : sous-échantillonnage des classes majoritaires (plafond à 2000) et augmentation ciblée des classes minoritaires (cible à 800). Les ensembles de validation et de test ont été conservés inchangés afin de préserver une évaluation réaliste.

## 6. Visualisation des images augmentées

In [ ]:
def show_augmented_samples(class_name: str, n: int = 6) -> None:
    class_dir = AUG_DIR / class_name
    if not class_dir.exists():
        print(f"Pas d'augmentations pour : {class_name}")
        return
    files = [f for f in class_dir.iterdir()
             if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
    if not files:
        print(f"Dossier vide : {class_dir}")
        return

    sample = random.sample(files, min(n, len(files)))
    cols = 3
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), squeeze=False)
    for i, f in enumerate(sample):
        ax = axes[i // cols][i % cols]
        with Image.open(f) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(f.name, fontsize=9)
        ax.axis("off")
    for j in range(len(sample), rows * cols):
        axes[j // cols][j % cols].axis("off")
    plt.suptitle(f"Augmentations — {class_name}", fontsize=13)
    plt.tight_layout()
    plt.show()

if AUG_DIR.exists():
    aug_classes = sorted([d.name for d in AUG_DIR.iterdir() if d.is_dir()])
    print("Classes augmentées :", aug_classes)
    for cls in aug_classes:
        show_augmented_samples(cls, n=6)
else:
    print("Pas de dossier augmented_train — aucune classe n'était sous le seuil.")

## 7. Class weights

Poids calculés par `compute_class_weight("balanced")` — utilisables directement dans `CrossEntropyLoss(weight=...)`. Plus le poids est élevé, plus la classe est rare dans le train final.

In [ ]:
weights_path = OUTPUT_DIR / "class_weights.csv"
if weights_path.exists():
    weights_df = pd.read_csv(weights_path).sort_values("weight", ascending=True)
    display(weights_df)

    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(weights_df))))
    ax.barh(weights_df["class_name"], weights_df["weight"], color="#7B68EE")
    ax.axvline(1.0, color="gray", linestyle="--", linewidth=1,
               label="poids neutre = 1.0")
    ax.set_xlabel("Poids (balanced)")
    ax.set_title("Class weights — plus élevé = classe plus rare")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f"Fichier absent : {weights_path}")

## 8. Vérification finale d'intégrité

Contrôles avant livraison du dataset :

1. Tous les chemins dans les CSV existent sur disque
2. Aucune image n'est corrompue (re-décodage PIL complet)
3. Aucun chevauchement entre train / val / test
4. Cohérence entre les CSV et l'arborescence `tomato_final/`
5. Mapping `label` ↔ `class_name` stable

Le contrôle 2 (re-décodage PIL) est le plus lent. Il est activé par défaut ici car le sens de cette étape est précisément de détecter une image qui serait passée à travers le filtre initial — désactivez-le via `DEEP_CHECK = False` si vous avez des millions d'images.

In [ ]:
DEEP_CHECK = True   # re-décodage PIL de chaque image (lent mais exhaustif)

report = {
    "missing_paths": [],
    "corrupted": [],
    "overlaps": {},
    "split_vs_tree_mismatch": {},
    "label_inconsistencies": [],
}

all_splits = {"train": train_df, "val": val_df, "test": test_df}

### 8.1 Existence des chemins + décodage PIL

In [ ]:
for split_name, df_split in all_splits.items():
    for fp in df_split["filepath"]:
        path = Path(fp)
        if not path.exists():
            report["missing_paths"].append((split_name, fp))
            continue
        if DEEP_CHECK:
            try:
                with Image.open(path) as im:
                    im.verify()
                with Image.open(path) as im:
                    im.load()
            except Exception as e:
                report["corrupted"].append((split_name, fp, str(e)))

print(f"Chemins manquants : {len(report['missing_paths'])}")
print(f"Images corrompues : {len(report['corrupted'])}")

if report["missing_paths"][:10]:
    print("\nExemples de chemins manquants :")
    for split, fp in report["missing_paths"][:10]:
        print(f"  [{split}] {fp}")

if report["corrupted"][:10]:
    print("\nExemples d'images corrompues :")
    for split, fp, err in report["corrupted"][:10]:
        print(f"  [{split}] {fp} — {err}")

### 8.2 Chevauchements entre splits

Même un seul chemin partagé entre train et test est une fuite de données.

In [ ]:
splits_sets = {name: set(df["filepath"]) for name, df in all_splits.items()}

for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    inter = splits_sets[a] & splits_sets[b]
    if inter:
        report["overlaps"][f"{a} ∩ {b}"] = len(inter)
        print(f"  {a} ∩ {b} : {len(inter)} chemin(s) en commun — FUITE")
        for fp in list(inter)[:5]:
            print(f"    {fp}")
    else:
        print(f"  {a} ∩ {b} : 0 — OK")

### 8.3 Cohérence CSV ↔ arborescence `tomato_final/`

Le nombre de fichiers dans `tomato_final/<split>/` doit correspondre au nombre de lignes du CSV correspondant.

In [ ]:
if FINAL_DIR.exists():
    for split_name in ["train", "val", "test"]:
        split_dir = FINAL_DIR / split_name
        if not split_dir.exists():
            print(f"  {split_name} : arborescence absente")
            continue
        tree_count = sum(
            1 for cls_dir in split_dir.iterdir() if cls_dir.is_dir()
            for f in cls_dir.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTS
        )
        csv_count = len(all_splits[split_name])
        if tree_count != csv_count:
            report["split_vs_tree_mismatch"][split_name] = {
                "csv": csv_count, "tree": tree_count
            }
            print(f"  {split_name} : CSV={csv_count} / arbre={tree_count} — INCOHÉRENT")
        else:
            print(f"  {split_name} : CSV={csv_count} / arbre={tree_count} — OK")
else:
    print(f"Arborescence absente : {FINAL_DIR}")

In [ ]:
# 8.3bis — Re-vérification PIL directement sur tomato_final/ (ce qu'on uploade)
final_corrupted = []
final_checked = 0

if FINAL_DIR.exists():
    for split_name in ["train", "val", "test"]:
        split_dir = FINAL_DIR / split_name
        if not split_dir.exists():
            continue
        for class_dir in split_dir.iterdir():
            if not class_dir.is_dir():
                continue
            for f in class_dir.iterdir():
                if not (f.is_file() and f.suffix.lower() in IMAGE_EXTS):
                    continue
                final_checked += 1
                try:
                    with Image.open(f) as im:
                        im.verify()
                    with Image.open(f) as im:
                        im.load()
                except Exception as e:
                    final_corrupted.append((split_name, str(f), str(e)))

print(f"Images vérifiées dans tomato_final/ : {final_checked}")
print(f"Images corrompues : {len(final_corrupted)}")

if final_corrupted[:10]:
    print("\nExemples :")
    for split, fp, err in final_corrupted[:10]:
        print(f"  [{split}] {fp} — {err}")
else:
    print("✅ Toutes les images de tomato_final/ sont décodables — prêt pour Glicid.")

### 8.4 Stabilité du mapping label ↔ class_name

Chaque `label` doit correspondre à exactement un `class_name` dans chaque split.

In [ ]:
for split_name, df_split in all_splits.items():
    if "label" not in df_split.columns:
        print(f"  {split_name} : colonne 'label' absente — skip")
        continue
    check = df_split.groupby("label")["class_name"].nunique()
    bad = check[check > 1]
    if not bad.empty:
        report["label_inconsistencies"].append((split_name, bad.to_dict()))
        print(f"  {split_name} : INCOHÉRENT — {bad.to_dict()}")
    else:
        print(f"  {split_name} : OK ({df_split['label'].nunique()} labels uniques)")

### Verdict final

In [ ]:
n_issues = (
    len(report["missing_paths"])
    + len(report["corrupted"])
    + len(report["overlaps"])
    + len(report["split_vs_tree_mismatch"])
    + len(report["label_inconsistencies"])
)

print("=" * 60)
if n_issues == 0:
    print("✅ Dataset VALIDE — aucune anomalie détectée.")
    print("   Prêt pour le transfert sur Glicid et l'entraînement.")
else:
    print(f"⚠  Dataset contient {n_issues} anomalie(s) :")
    if report["missing_paths"]:
        print(f"   - {len(report['missing_paths'])} chemin(s) manquant(s)")
    if report["corrupted"]:
        print(f"   - {len(report['corrupted'])} image(s) corrompue(s)")
    if report["overlaps"]:
        print(f"   - chevauchements : {report['overlaps']}")
    if report["split_vs_tree_mismatch"]:
        print(f"   - incohérence CSV/arbre : {report['split_vs_tree_mismatch']}")
    if report["label_inconsistencies"]:
        print(f"   - label↔class_name : {report['label_inconsistencies']}")
print("=" * 60)